In [0]:
import requests

workspace_url = "https://dbc-6727b40a-8ab7.cloud.databricks.com"
# Get token securely from Databricks context instead of hardcoding
token = dbutils.notebook.entry_point.getDbutils().notebook().getContext().apiToken().get()
space_id = "01f1a7cce8341affb459c8c51394741b"
org_id = "7474656177544306"

headers = {
    "Authorization": f"Bearer {token}",
    "Content-Type": "application/json"
}

response = requests.post(
    f"{workspace_url}/api/2.0/genie/spaces/{space_id}/start-conversation?o={org_id}",
    headers=headers,
    json={"content": "Hello"}
)

conversation = response.json()

print(conversation)

In [0]:
conversation_id = conversation["conversation_id"]

payload = {
    "content":
    "How many active employees do we have?"
}

response = requests.post(
    f"{workspace_url}/api/2.0/genie/spaces/{space_id}/conversations/{conversation_id}/messages?o={org_id}",
    headers=headers,
    json=payload
)

# Check if response is successful and contains JSON
if response.status_code == 200:
    print(response.json())
else:
    print(f"Error {response.status_code}: {response.text}")

In [0]:
import time

# Get the message ID from the previous response
message_id = response.json()["id"]

# Poll for the answer (Genie needs time to process)
max_attempts = 30
for attempt in range(max_attempts):
    time.sleep(2)  # Wait 2 seconds between polls
    
    # Get the message details
    get_response = requests.get(
        f"{workspace_url}/api/2.0/genie/spaces/{space_id}/conversations/{conversation_id}/messages/{message_id}?o={org_id}",
        headers=headers
    )
    
    if get_response.status_code == 200:
        message_data = get_response.json()
        status = message_data.get("status")
        
        print(f"Attempt {attempt + 1}: Status = {status}")
        
        if status == "COMPLETED":
            # Extract the answer
            attachments = message_data.get("attachments", [])
            if attachments:
                for attachment in attachments:
                    if attachment.get("text"):
                        print("\n=== GENIE'S ANSWER ===")
                        print(attachment["text"]["content"])
                        break
            else:
                print("\nNo answer attachments found")
                print(f"Full message data: {message_data}")
            break
        elif status == "FAILED":
            print(f"\nQuery failed: {message_data}")
            break
    else:
        print(f"Error retrieving message: {get_response.status_code} - {get_response.text}")
        break
else:
    print("\nTimeout: Answer not received within expected time")

In [0]:
payload = {
    "content":
    "Which region has the most active employees?"
}

response = requests.post(
    f"{workspace_url}/api/2.0/genie/spaces/{space_id}/conversations/{conversation_id}/messages?o={org_id}",
    headers=headers,
    json=payload
)

if response.status_code == 200:
    print(response.json())
else:
    print(f"Error {response.status_code}: {response.text}")

In [0]:
# Get the message ID from Cell 4's response
message_id = response.json()["id"]

# Poll for the completed message with query details
import time
max_attempts = 30
for attempt in range(max_attempts):
    time.sleep(2)
    
    get_response = requests.get(
        f"{workspace_url}/api/2.0/genie/spaces/{space_id}/conversations/{conversation_id}/messages/{message_id}?o={org_id}",
        headers=headers
    )
    
    if get_response.status_code == 200:
        message_data = get_response.json()
        status = message_data.get("status")
        
        print(f"Attempt {attempt + 1}: Status = {status}")
        
        if status == "COMPLETED":
            # Extract the generated SQL from attachments
            attachments = message_data.get("attachments", [])
            for attachment in attachments:
                if "query" in attachment:
                    query_info = attachment["query"]
                    print("\n=== GENERATED SQL ===")
                    print(query_info.get("query", "No SQL found"))
                    print("\n=== DESCRIPTION ===")
                    print(query_info.get("description", "No description"))
                    break
            break
        elif status == "FAILED":
            print(f"\nQuery failed: {message_data}")
            break
else:
    print("\nTimeout: Query not completed within expected time")